# Geospatial Pipeline Demo

`analytics_toolbox.geospatial` — offline US address-to-block-group geocoding.

This notebook walks through the full pipeline end-to-end:
1. One-time reference data ingestion (NAD + TIGER)
2. Address normalization
3. NAD fuzzy matching
4. Block group assignment
5. Inspecting results and handling edge cases

**Privacy note:** no address data leaves the machine. All reference data is downloaded from `.gov` domains once and stored locally in DuckDB.

## 0. Setup

Install the package (skip if already installed):
```bash
pip install analytics-toolbox
```

In [18]:
from pathlib import Path

import pandas as pd

from analytics_toolbox.geospatial import geocode_address_table, normalize_addresses
from analytics_toolbox.geospatial._config import load_config
from analytics_toolbox.geospatial.address_geocoder import geocode_addresses, ingest_tiger
from analytics_toolbox.geospatial.address_matcher import match_addresses
from analytics_toolbox.geospatial.nad_preprocess_ingest import ingest_nad

## 1. Config

Create a `config.yaml` file (or point to an existing one). The config controls which states to ingest, where DuckDB lives, and the confidence threshold.

In [19]:
CONFIG_YAML = """
nad:
  states: [IA]
  force_refresh: false

tiger:
  vintage: 2024
  force_refresh: false

matching:
  confidence_threshold: 90

storage:
  data_dir: ~/.local/share/analytics_toolbox/
  connection: ~/.local/share/analytics_toolbox/analytics_toolbox.duckdb
"""

config_path = Path("/tmp/demo_config.yaml")
config_path.write_text(CONFIG_YAML)
config = load_config(config_path)
print("Config loaded.")
print(f"  States: {config.nad.states}")
print(f"  DuckDB: {config.storage.connection}")
print(f"  Confidence threshold: {config.matching.confidence_threshold}")

Config loaded.
  States: ['IA']
  DuckDB: /Users/matthewbargstadt/.local/share/analytics_toolbox/analytics_toolbox.duckdb
  Confidence threshold: 90


## 2. One-time reference data ingestion

Run each cell once to populate the local DuckDB. Subsequent runs skip if data already exists.

- **TIGER**: ~50 MB Iowa block group + ZCTA shapefiles from `census.gov` — run **first**
- **NAD**: ~8.5 GB national ZIP from `datahub.transportation.gov` (downloaded once, filtered to IA locally)

**Ingest order matters:** `ingest_tiger` first, then `ingest_nad`. The NAD ingest uses TIGER ZCTA polygons to impute missing postal codes on NAD records that have lat/lon but no ZIP. If NAD runs first, imputation is skipped for that state — re-run NAD with `force_refresh: true` after TIGER is loaded to pick it up.

**Disk space**: plan for ~25 GB free during NAD ingest (ZIP + extracted text).

Or via CLI:
```bash
analytics-toolbox ingest-tiger --config config.yaml
analytics-toolbox ingest-nad --config config.yaml
```

In [20]:
# Step 1 of 2: Download TIGER block group + ZCTA shapefiles for Iowa — run this first.
# TIGER ZCTA polygons are used by the NAD ingest to impute postal codes on NAD records
# that have lat/lon but no ZIP code.
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")

ingest_tiger(config)

[tiger] downloading block group shapefiles (vintage 2024)
[tiger] downloading block groups for IA (FIPS 19)
HTTP Request: GET https://www2.census.gov/geo/tiger/TIGER2024/BG/tl_2024_19_bg.zip "HTTP/1.1 200 OK"
[tiger] downloading ZCTA shapefile (vintage 2024)
HTTP Request: GET https://www2.census.gov/geo/tiger/TIGER2024/ZCTA520/tl_2024_us_zcta520.zip "HTTP/1.1 200 OK"
[tiger] loaded 2703 block groups into tiger_block_groups_2024
[tiger] loaded 33791 ZCTA polygons into tiger_zcta_2024
[tiger] done — tiger_block_groups_2024 and tiger_zcta_2024 loaded


In [21]:
# Step 2 of 2: Download the NAD national file and ingest Iowa rows.
# First run downloads ~8.5 GB from datahub.transportation.gov (one-time;
# future state ingests reuse the cached national file).
# After normalization, postal codes are imputed for any Iowa NAD records
# with lat/lon but no ZIP using the TIGER ZCTA polygons loaded above.
ingest_nad(config)

[nad] downloading national ZIP (~8.5 GB) from https://datahub.transportation.gov/api/views/fc2s-wawr/files/3a27e4dd-7fe4-42c8-bd58-1e81a8de6ab9?filename=TXT.zip — one-time download; future state ingests reuse this file
HTTP Request: GET https://datahub.transportation.gov/api/views/fc2s-wawr/files/3a27e4dd-7fe4-42c8-bd58-1e81a8de6ab9?filename=TXT.zip "HTTP/1.1 200 OK"
[nad] download complete: /Users/matthewbargstadt/.local/share/analytics_toolbox/nad_national.zip
[nad] extracting national ZIP to /Users/matthewbargstadt/.local/share/analytics_toolbox/nad_national.txt
[nad] extracting entry 'TXT/NAD_r22.txt' (40.7 GB uncompressed) — this may take several minutes
[nad] extraction progress: 1 GB written
[nad] extraction progress: 2 GB written
[nad] extraction progress: 3 GB written
[nad] extraction progress: 4 GB written
[nad] extraction progress: 5 GB written
[nad] extraction progress: 6 GB written
[nad] extraction progress: 7 GB written
[nad] extraction progress: 8 GB written
[nad] extrac

## 3. Prepare sample addresses

Six Iowa addresses with intentional deviations from how they appear in NAD — the kind of real-world inconsistencies the fuzzy matcher is designed to handle:

| # | Deviation type | Detail |
|---|---|---|
| 1 | Spelled-out directional + street type | "East Grand Avenue" vs NAD's "E Grand Ave" |
| 2 | Street name typo | "Ingersol" vs NAD's "Ingersoll" (one missing 'l') |
| 3 | Missing directional | "Clinton St" vs NAD's "S Clinton St" |
| 4 | Written-out ordinal | "Third Ave SE" vs NAD's "3rd Ave SE" |
| 5 | PO Box | Non-standard → postal centroid fallback |
| 6 | Military (APO/AE) | Non-standard → postal centroid fallback |

In [22]:
sample = pd.DataFrame([
    # Deviation: spelled-out directional + street type (NAD: "215 E Grand Ave")
    {"id": 1, "Street_Address": "215 East Grand Avenue",
     "City": "Des Moines", "State": "IA", "Postal_Code": "50309"},
    # Deviation: typo — "Ingersol" missing one 'l' (NAD: "Ingersoll")
    {"id": 2, "Street_Address": "4321 Ingersol Ave",
     "City": "Des Moines", "State": "IA", "Postal_Code": "50312"},
    # Deviation: dropped directional (NAD: "123 S Clinton St")
    {"id": 3, "Street_Address": "123 Clinton St",
     "City": "Iowa City", "State": "IA", "Postal_Code": "52240"},
    # Deviation: written-out ordinal (NAD: "500 3rd Ave SE")
    {"id": 4, "Street_Address": "500 Third Ave SE",
     "City": "Cedar Rapids", "State": "IA", "Postal_Code": "52401"},
    # Non-standard: PO Box → postal centroid fallback
    {"id": 5, "Street_Address": "PO Box 4500",
     "City": "Ames", "State": "IA", "Postal_Code": "50010"},
    # Non-standard: military address (APO/AE) → postal centroid fallback
    {"id": 6, "Street_Address": "Unit 2B",
     "City": "APO", "State": "AE", "Postal_Code": "09001"},
])

sample

,id,Street_Address,City,State,Postal_Code
0,1,215 East Grand Avenue,Des Moines,IA,50309
1,2,4321 Ingersol Ave,Des Moines,IA,50312
2,3,123 Clinton St,Iowa City,IA,52240
3,4,500 Third Ave SE,Cedar Rapids,IA,52401
4,5,PO Box 4500,Ames,IA,50010
5,6,Unit 2B,APO,AE,09001


## 4. Full pipeline: single call

`geocode_address_table` chains normalize → match → geocode in one shot.

In [23]:
result = geocode_address_table(sample, config_path)

display_cols = [
    "id", "Street_Address", "Postal_Code",
    "normalized_address_line_1", "is_standard_address", "address_flag",
    "match_method", "match_score",
    "matched_latitude", "matched_longitude",
    "block_group_fips", "census_tract_fips",
    "location_imputed",
]

result
# result[display_cols].T

No ZCTA centroid for postal code 09001


,id,Street_Address,City,State,Postal_Code,normalized_address_line_1,normalized_address_line_2,normalized_city,normalized_state,normalized_postal_code,...,match_rank,match_method,matched_latitude,matched_longitude,matched_state_fips,matched_county_fips,block_group_fips,census_tract_fips,tiger_vintage,location_imputed
0,1,215 East Grand Avenue,Des Moines,IA,50309,215 E GRAND AVE,None,DES MOINES,IA,50309,...,1,nad_match,41.590699,-93.610456,Po,Polk,191530051021,19153005102,2024,False
1,2,4321 Ingersol Ave,Des Moines,IA,50312,4321 INGERSOL AVE,None,DES MOINES,IA,50312,...,1,nad_match,41.586854,-93.672117,Po,Polk,191530029003,19153002900,2024,False
2,3,123 Clinton St,Iowa City,IA,52240,123 CLINTON ST,None,IOWA CITY,IA,52240,...,1,postal_centroid,41.631536,-91.499230,NaN,NaN,191030018012,19103001801,2024,True
3,4,500 Third Ave SE,Cedar Rapids,IA,52401,500 THIRD AVE SE,None,CEDAR RAPIDS,IA,52401,...,1,postal_centroid,41.975484,-91.658769,NaN,NaN,191130027002,19113002700,2024,True
4,5,PO Box 4500,Ames,IA,50010,NaN,None,NaN,NaN,NaN,...,1,non_standard,42.033223,-93.587622,NaN,NaN,191690009003,19169000900,2024,True
5,6,Unit 2B,APO,AE,09001,NaN,None,NaN,NaN,NaN,...,1,non_standard,NaN,NaN,NaN,NaN,NaN,NaN,2024,True


## 5. Step-by-step walkthrough

Running the pipeline steps individually is useful for debugging and understanding what each step contributes.

### Step 1: Normalize

In [24]:
normalized = normalize_addresses(sample)

normalizer_cols = [
    "id", "Street_Address",
    "normalized_address_line_1",
    "normalized_postal_code",
    "is_standard_address",
    "address_flag",
    "normalization_note",
]
normalized[normalizer_cols]

,id,Street_Address,normalized_address_line_1,normalized_postal_code,is_standard_address,address_flag,normalization_note
0,1,215 East Grand Avenue,215 E GRAND AVE,50309,True,standard,NaN
1,2,4321 Ingersol Ave,4321 INGERSOL AVE,50312,True,standard,NaN
2,3,123 Clinton St,123 CLINTON ST,52240,True,standard,NaN
3,4,500 Third Ave SE,500 THIRD AVE SE,52401,True,standard,NaN
4,5,PO Box 4500,NaN,NaN,False,UnParseableAddressError,UNPARSEABLE ADDRESS: Unable to break this addr...
5,6,Unit 2B,NaN,NaN,False,UnParseableAddressError,UNPARSEABLE ADDRESS: Unable to break this addr...


In [25]:
# Non-standard addresses are flagged here and will skip street-level matching downstream
non_standard = normalized[~normalized["is_standard_address"]]
print(f"{len(non_standard)} non-standard addresses (will use postal centroid fallback):")
print(non_standard[["Street_Address", "address_flag", "normalization_note"]].to_string())

2 non-standard addresses (will use postal centroid fallback):
  Street_Address             address_flag                                                                                                                                                                                                                    normalization_note
4    PO Box 4500  UnParseableAddressError  UNPARSEABLE ADDRESS: Unable to break this address into its component parts, OrderedDict([('address_line_1', 'PO BOX 4500, AMES, IA, 50010'), ('address_line_2', None), ('city', 'Ames'), ('state', 'IA'), ('postal_code', '50010')])
5        Unit 2B  UnParseableAddressError        UNPARSEABLE ADDRESS: Unable to break this address into its component parts, OrderedDict([('address_line_1', 'UNIT 2B, APO, AE, 09001'), ('address_line_2', None), ('city', 'APO'), ('state', 'AE'), ('postal_code', '09001')])


### Step 2: Match against NAD

In [26]:
matched = match_addresses(normalized, config)

matcher_cols = [
    "id", "Street_Address",
    "match_method", "match_score", "match_rank",
    "nad_id",
    "matched_latitude", "matched_longitude",
    "matched_state_fips", "matched_county_fips",
]
matched[matcher_cols]

No ZCTA centroid for postal code 09001


,id,Street_Address,match_method,match_score,match_rank,nad_id,matched_latitude,matched_longitude,matched_state_fips,matched_county_fips
0,1,215 East Grand Avenue,nad_match,93.333336,1,16892822,41.590699,-93.610456,Po,Polk
1,2,4321 Ingersol Ave,nad_match,91.428574,1,16903397,41.586854,-93.672117,Po,Polk
2,3,123 Clinton St,postal_centroid,NaN,1,NaN,41.631536,-91.499230,NaN,NaN
3,4,500 Third Ave SE,postal_centroid,NaN,1,NaN,41.975484,-91.658769,NaN,NaN
4,5,PO Box 4500,non_standard,NaN,1,NaN,42.033223,-93.587622,NaN,NaN
5,6,Unit 2B,non_standard,NaN,1,NaN,NaN,NaN,NaN,NaN


In [27]:
# Summary: how many addresses matched at street level vs fell back to centroid?
matched["match_method"].value_counts()

match_method
nad_match          2
postal_centroid    2
non_standard       2
Name: count, dtype: int64

### Step 3: Geocode to block group

In [28]:
geocoded = geocode_addresses(matched, config)

geocoder_cols = [
    "id", "Street_Address",
    "match_method", "location_imputed",
    "block_group_fips", "census_tract_fips",
    "tiger_vintage",
]
geocoded[geocoder_cols]

,id,Street_Address,match_method,location_imputed,block_group_fips,census_tract_fips,tiger_vintage
0,1,215 East Grand Avenue,nad_match,False,191530051021,19153005102,2024
1,2,4321 Ingersol Ave,nad_match,False,191530029003,19153002900,2024
2,3,123 Clinton St,postal_centroid,True,191030018012,19103001801,2024
3,4,500 Third Ave SE,postal_centroid,True,191130027002,19113002700,2024
4,5,PO Box 4500,non_standard,True,191690009003,19169000900,2024
5,6,Unit 2B,non_standard,True,NaN,NaN,2024


## 6. Top-N matching for human adjudication

Set `top_n > 1` to return multiple NAD candidates per input row. Useful when you want a human to confirm ambiguous matches. Each candidate gets a `match_rank` column (1 = best).

In [29]:
# Match the first address with top_n=3 to see all close NAD candidates
single = normalized.iloc[[0]]
top3 = match_addresses(single, config, top_n=3)

top3[[
    "Street_Address", "match_rank", "match_score", "match_method",
    "nad_id", "matched_latitude", "matched_longitude",
]]

,Street_Address,match_rank,match_score,match_method,nad_id,matched_latitude,matched_longitude
0,215 East Grand Avenue,1,93.333336,nad_match,16892822,41.590699,-93.610456
0,215 East Grand Avenue,2,93.333336,nad_match,16962846,41.589868,-93.614405
0,215 East Grand Avenue,3,93.333336,nad_match,16860612,41.589851,-93.614213


In [30]:
# Filter back to top-1 for the automated path
top1_only = top3[top3["match_rank"] == 1]
print(f"Rows after filtering to rank 1: {len(top1_only)}")

Rows after filtering to rank 1: 1


## 7. Understanding the output flags

Key columns to understand when consuming this output downstream:

In [31]:
# is_standard_address: only True for clean street addresses
# location_imputed: True whenever lat/lon is a postal centroid rather than a street point
# block_group_fips may be None for addresses in states not in your NAD ingest,
#   or for addresses that fall outside TIGER polygon coverage (rare)

flag_summary = result.groupby(["is_standard_address", "match_method", "location_imputed"]).size()
flag_summary.name = "count"
flag_summary.reset_index()

,is_standard_address,match_method,location_imputed,count
0,False,non_standard,True,2
1,True,nad_match,False,2
2,True,postal_centroid,True,2


In [32]:
# For downstream analysis: always include location_imputed so consumers know
# which coordinates are street-level vs postal-centroid imputed
print("Rows with imputed location (postal centroid, not street point):")
imputed_rows = result[result["location_imputed"]]
print(imputed_rows[["id", "Street_Address", "match_method", "block_group_fips"]])

Rows with imputed location (postal centroid, not street point):
   id    Street_Address     match_method block_group_fips
2   3    123 Clinton St  postal_centroid     191030018012
3   4  500 Third Ave SE  postal_centroid     191130027002
4   5       PO Box 4500     non_standard     191690009003
5   6           Unit 2B     non_standard              NaN


## 8. Force-refresh reference data

Re-run ingestion to pick up a new NAD release or TIGER vintage. The `force_refresh` flag deletes existing state rows before re-ingesting, so partial state refreshes don't orphan rows from other states.

Run TIGER first so the fresh ZCTA polygons are available for postal code imputation during NAD re-ingest:

```python
from analytics_toolbox.geospatial._config import NadConfig, TigerConfig
import dataclasses

# Refresh TIGER first (picks up new vintage or geom_wkt column if previously missing)
tiger_refresh_config = dataclasses.replace(
    config,
    tiger=TigerConfig(vintage=2024, force_refresh=True),
)
ingest_tiger(tiger_refresh_config)

# Then refresh Iowa NAD — will impute postal codes using the fresh TIGER data
ia_refresh_config = dataclasses.replace(
    config,
    nad=NadConfig(states=["IA"], force_refresh=True),
)
ingest_nad(ia_refresh_config)
```

Or via CLI:
```bash
analytics-toolbox ingest-tiger --config config.yaml --force-refresh
analytics-toolbox ingest-nad --config config.yaml --force-refresh
```

## 9. MotherDuck (managed DuckDB)

Switch from a local DuckDB file to MotherDuck by changing a single config line — no code changes required:

```yaml
storage:
  data_dir: ~/.local/share/analytics_toolbox/
  connection: md:analytics_toolbox   # ← MotherDuck connection string
```

Authenticate first: `! motherduck auth` (or set `motherduck_token` in your environment).

The API is identical — the same `load_config`, `ingest_nad`, `ingest_tiger`, and `geocode_address_table` calls work against MotherDuck without modification.

## 10. Stress test — pipeline throughput at scale

Samples real Iowa NAD addresses directly from DuckDB, applies random deviations (spelling out abbreviations, dropping directionals, introducing typos) to simulate messy real-world input, then times each pipeline stage.

Change `--n` to `1000000` for the full 1M-row run. The match stage is the bottleneck — it's O(unique postal codes) in DuckDB queries after the postal-code caching optimization, then pure RapidFuzz scoring in memory.

In [33]:
# Quick run: 10K addresses (~seconds). Change to 1_000_000 for the full stress test.
!python ../benchmarks/stress_test_iowa.py --config /tmp/demo_config.yaml --n 100000

Stress test: 100,000 Iowa addresses
DuckDB: /Users/matthewbargstadt/.local/share/analytics_toolbox/analytics_toolbox.duckdb
Sampling 100,000 addresses from nad_addresses (state = 'IA')...
  Sampled 100,000 rows in 0.1s
Applying random deviations...
  Deviations applied in 0.1s

[1/3] normalize_addresses  (100,000 rows)
      done in 4.9s  (20,589 rows/sec)

[2/3] match_addresses  (100,000 rows)
      Candidates fetched once per unique postal code, reused across rows.
No ZCTA centroid for postal code 0
No ZCTA centroid for postal code 00000
No ZCTA centroid for postal code 50765
      done in 832.1s  (120 rows/sec)

[3/3] geocode_addresses  (100,000 rows)
      done in 0.5s  (211,223 rows/sec)

STRESS TEST SUMMARY
Total rows:                         100,000
normalize_addresses:                  4.9s  (20,589 rows/sec)
match_addresses:                    13.9m  (120 rows/sec)
geocode_addresses:                    0.5s  (211,223 rows/sec)
Total pipeline:                     14.0m  (119 ro